In [2]:
import gc
import itertools
import os
import random
import sys
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Literal, TypeAlias

import circuitsvis as cv
import einops
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import requests
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from IPython.display import HTML, IFrame, display
from jaxtyping import Float, Int
from openai import OpenAI
from rich import print as rprint
from rich.table import Table
from sae_lens import (
    SAE,
    ActivationsStore,
    GatedTrainingSAEConfig,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    LoggingConfig,
)
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory
from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
from tabulate import tabulate
from torch import Tensor
from tqdm.auto import tqdm
from transformer_lens import ActivationCache
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, test_prompt

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


def _get_hook_layer(sae: SAE) -> int:
  """Extract the layer number from an SAE's hook name (e.g. 'blocks.7.hook_resid_pre' → 7)."""
  return int(sae.cfg.metadata.hook_name.split(".")[1])

def display_dashboard(
  sae_release="gpt2-small-res-jb",
  sae_id="blocks.7.hook_resid_pre",
  latent_idx=0,
  width=800,
  height=600,
):
  release = get_pretrained_saes_directory()[sae_release]
  neuronpedia_id = release.neuronpedia_id[sae_id]

  url = f"https://neuronpedia.org/{neuronpedia_id}/{latent_idx}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

  print(url)
  display(IFrame(url, width=width, height=height))

In [3]:
t.set_grad_enabled(False)

gpt2: HookedSAETransformer = HookedSAETransformer.from_pretrained("gpt2-small", device=device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer


In [ ]:
attn_saes = {
  layer: SAE.from_pretrained(
    "gpt2-small-hook-z-kk",
    f"blocks.{layer}.hook_z",
    device=str(device),
  )
  for layer in range(gpt2.cfg.n_layers)
}

In [20]:
names = [" John", " Mary"]
name_tokens = [gpt2.to_single_token(name) for name in names]

prompt_template = "When{A} and{B} went to the shops,{S} gave the bag to"
prompts = [
    prompt_template.format(A=names[i], B=names[1 - i], S=names[j]) for i, j in itertools.product(range(2), range(2))
]
rprint(prompts)
correct_answers = names[::-1] * 2
incorrect_answers = names * 2
correct_toks = gpt2.to_tokens(correct_answers, prepend_bos=False)[:, 0].tolist()
incorrect_toks = gpt2.to_tokens(incorrect_answers, prepend_bos=False)[:, 0].tolist()


def logits_to_ave_logit_diff(
    logits: Float[Tensor, "batch seq d_vocab"],
    correct_toks: list[int] = correct_toks,
    incorrect_toks: list[int] = incorrect_toks,
    reduction: Literal["mean", "sum"] | None = "mean",
    keep_as_tensor: bool = False,
) -> list[float] | float:
    """
    Returns the avg logit diff on a set of prompts, with fixed s2 pos and stuff.
    """
    correct_logits = logits[range(len(logits)), -1, correct_toks]
    incorrect_logits = logits[range(len(logits)), -1, incorrect_toks]
    logit_diff = correct_logits - incorrect_logits
    if reduction is not None:
        logit_diff = logit_diff.mean() if reduction == "mean" else logit_diff.sum()
    return logit_diff if keep_as_tensor else logit_diff.tolist()


# Testing a single prompt (where correct answer is John), verifying model gets it right
test_prompt(prompts[1], names, gpt2)

# Testing logits over all 4 prompts, verifying the model always has a high logit diff
logits = gpt2(prompts, return_type="logits")
logit_diffs = logits_to_ave_logit_diff(logits, reduction=None)
print(
    tabulate(
        zip(prompts, correct_answers, logit_diffs),
        headers=["Prompt", "Answer", "Logit Diff"],
        tablefmt="simple_outline",
        numalign="left",
        floatfmt="+.3f",
    )
)


[
    'When John and Mary went to the shops, John gave the bag to',
    'When John and Mary went to the shops, Mary gave the bag to',
    'When Mary and John went to the shops, John gave the bag to',
    'When Mary and John went to the shops, Mary gave the bag to'
]

Tokenized prompt: ['<|endoftext|>', 'When', ' John', ' and', ' Mary', ' went', ' to', ' the', ' shops', ',', ' Mary', ' gave', ' the', ' bag', ' to']
Tokenized answers: [[' John'], [' Mary']]


Performance on answer tokens:
Rank: 0        Logit: 18.03 Prob: 69.35% Token: | John|
Rank: 3        Logit: 14.83 Prob:  2.82% Token: | Mary|

Top 0th token. Logit: 18.03 Prob: 69.35% Token: | John|
Top 1th token. Logit: 15.53 Prob:  5.67% Token: | them|
Top 2th token. Logit: 15.28 Prob:  4.42% Token: | the|
Top 3th token. Logit: 14.83 Prob:  2.82% Token: | Mary|
Top 4th token. Logit: 14.16 Prob:  1.44% Token: | her|
Top 5th token. Logit: 13.94 Prob:  1.16% Token: | him|
Top 6th token. Logit: 13.72 Prob:  0.93% Token: | a|
Top 7th token. Logit: 13.68 Prob:  0.89% Token: | Joseph|
Top 8th token. Logit: 13.61 Prob:  0.83% Token: | Jesus|
Top 9th token. Logit: 13.34 Prob:  0.64% Token: | their|


Ranks of the answer tokens: [[(' John', 0), (' Mary', 3)]]

┌────────────────────────────────────────────────────────────┬──────────┬──────────────┐
│ Prompt                                                     │ Answer   │ Logit Diff   │
├────────────────────────────────────────────────────────────┼──────────┼──────────────┤
│ When John and Mary went to the shops, John gave the bag to │ Mary     │ +3.337       │
│ When John and Mary went to the shops, Mary gave the bag to │ John     │ +3.202       │
│ When Mary and John went to the shops, John gave the bag to │ Mary     │ +3.918       │
│ When Mary and John went to the shops, Mary gave the bag to │ John     │ +2.220       │
└────────────────────────────────────────────────────────────┴──────────┴──────────────┘


In [ ]:
clean_logit_diff = logit_diffs
rprint(clean_logit_diff)

In [ ]:
logits_diff = [logits_to_ave_logit_diff(gpt2(prompts, return_type="logits"), reduction=None, keep_as_tensor=True)]

In [ ]:
for layer in range(0,12):
  with gpt2.saes(saes=[attn_saes[layer]]):
    logits_diff.append(logits_to_ave_logit_diff(gpt2(prompts, return_type="logits"), reduction=None, keep_as_tensor=True))

In [ ]:
logit_diff = t.stack(logits_diff)
avg_diff = logit_diff.mean(dim=1)
perc_of_clean = [f"{perc.to(t.int)} %" for perc in ((avg_diff / avg_diff[0]) * 100)]
labels = ["clean"] + [f"layer{layer}" for layer in range(0, 12)]

print(
  tabulate(
    zip(labels, avg_diff, perc_of_clean),
      headers=["ablation", "logit_diff", "% of clean"],
      numalign="left",
      tablefmt="simple_outline",
      floatfmt="+.3f",
  )
)

In [24]:
_, cache = gpt2.run_with_cache(prompts, remove_batch_dim=True)

img1 = cache["pattern", 9][6].cpu().numpy()
img2 = cache["pattern", 9][9].cpu().numpy()  # example second head

fig = make_subplots(rows=1, cols=2, subplot_titles=("Head 6", "Head 9"))

fig.add_trace(go.Heatmap(z=img1, colorscale="Viridis"), row=1, col=1)
fig.add_trace(go.Heatmap(z=img2, colorscale="Viridis"), row=1, col=2)

size = 500
fig.update_layout(height=size, width=size*2)
fig.show()

In [4]:
layer9_sae = SAE.from_pretrained(
        "gpt2-small-hook-z-kk",
        f"blocks.{9}.hook_z",
        device=str(device),
    )

names = [" John", " Mary"]
name_tokens = [gpt2.to_single_token(name) for name in names]

prompt_template = "When{A} and{B} went to the shops,{S} gave the bag to"
prompts = [
    prompt_template.format(A=names[i], B=names[1 - i], S=names[j]) for i, j in itertools.product(range(2), range(2))
]
rprint(prompts)

[
    'When John and Mary went to the shops, John gave the bag to',
    'When John and Mary went to the shops, Mary gave the bag to',
    'When Mary and John went to the shops, John gave the bag to',
    'When Mary and John went to the shops, Mary gave the bag to'
]

In [40]:
_, cache = gpt2.run_with_cache_with_saes(prompts, saes=[layer9_sae])
sae_acts_post = cache[f"{layer9_sae.cfg.metadata.hook_name}.hook_sae_acts_post"][:, -1].mean(0)

rprint(cache[f"{layer9_sae.cfg.metadata.hook_name}.hook_sae_acts_post"][:, -1].mean(0).topk(3))

px.line(
  sae_acts_post.cpu().numpy(),
  title=f"Activations at the final token position ({sae_acts_post.nonzero().numel()} alive)",
  labels={"index": "Latent", "value": "Activation"},
  template="ggplot2",
  width=1000,
).update_layout(showlegend=False).show()


torch.return_types.topk(
values=tensor([8.2069, 5.5789, 1.6027]),
indices=tensor([11368, 18767,  3101]))

In [7]:
for act, ind in zip(*sae_acts_post.topk(3)):
  print(f"Latent {ind} had activation {act:.2f}")
  display_dashboard(
    sae_release="gpt2-small-hook-z-kk",
    sae_id=f"blocks.{9}.hook_z",
    latent_idx=int(ind),
  )

Latent 11368 had activation 8.21
https://neuronpedia.org/gpt2-small/9-att-kk/11368?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


Latent 18767 had activation 5.58
https://neuronpedia.org/gpt2-small/9-att-kk/18767?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


Latent 3101 had activation 1.60
https://neuronpedia.org/gpt2-small/9-att-kk/3101?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


In [31]:
layer9_sae.W_dec.shape

torch.Size([24576, 768])

In [52]:
feats = [11368, 18767,  3101]
feat_dec_w = einops.rearrange(layer9_sae.W_dec[feats], "feats (n_heads d_head) -> feats n_heads d_head", n_heads=gpt2.cfg.n_heads) 

norm_per_head = (feat_dec_w**2).sum(-1).sqrt()
norm_frac_per_head = norm_per_head / norm_per_head.sum(-1, keepdim=True)

table = Table("Head", *[f"Feature {i}" for i in feats])
for i in range(gpt2.cfg.n_heads):
    table.add_row(
        f"9.{i}", *[f"{frac:.2%}" for frac in norm_frac_per_head[:, i].tolist()]
    )

rprint(table)

┏━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Head ┃ Feature 11368 ┃ Feature 18767 ┃ Feature 3101 ┃
┡━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ 9.0  │ 6.91%         │ 5.36%         │ 13.85%       │
│ 9.1  │ 2.77%         │ 2.71%         │ 2.46%        │
│ 9.2  │ 7.00%         │ 5.00%         │ 11.17%       │
│ 9.3  │ 4.51%         │ 4.29%         │ 10.64%       │
│ 9.4  │ 5.43%         │ 6.16%         │ 2.98%        │
│ 9.5  │ 4.95%         │ 3.84%         │ 10.97%       │
│ 9.6  │ 23.40%        │ 13.17%        │ 3.06%        │
│ 9.7  │ 6.81%         │ 4.97%         │ 8.46%        │
│ 9.8  │ 10.62%        │ 6.14%         │ 26.64%       │
│ 9.9  │ 22.33%        │ 42.66%        │ 3.50%        │
│ 9.10 │ 2.35%         │ 2.23%         │ 3.88%        │
│ 9.11 │ 2.92%         │ 3.47%         │ 2.39%        │
└──────┴───────────────┴───────────────┴──────────────┘

In [ ]:
cache[f"{layer9_sae.cfg.metadata.hook_name}.hook_sae_acts_post"][:, -1, 11368]

tensor([13.0304,  3.1632, 14.1027,  2.5315])